# Jmail Network — Jeffrey Epstein Email Network Analysis

Interactive visualizations of the Epstein email network scraped from Jmail (Gmail emulator).  
Data includes **~1,533 nodes** and **~2,132 edges** with email weights and timestamps.

---
**Visualizations included:**
1. Interactive full network graph (pyvis)
2. Top contacts of Jeffrey Epstein
3. Activity timeline — new contacts & email volume by year
4. Degree distribution
5. Community detection
6. Ego network — Epstein's direct circle
7. Connection strength heatmap (top contacts)


In [ ]:
# ── 0. Install missing packages ──────────────────────────────────────────────
import subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

for pkg, name in [('pyvis', 'pyvis'), ('plotly', 'plotly'), ('networkx', 'networkx')]:
    try:
        __import__(name)
    except ImportError:
        print(f'Installing {pkg}...')
        pip_install(pkg)

print('All packages ready.')

In [ ]:
# ── 1. Imports ────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import networkx as nx
from collections import Counter
from IPython.display import display, HTML, IFrame
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pyvis.network import Network
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded.')

In [ ]:
# ── 2. Load Data ──────────────────────────────────────────────────────────────
nodes_df = pd.read_csv('cleaned_nodes.csv')
edges_df = pd.read_csv('cleaned_edges.csv')

# Parse timestamps
edges_df['first_interaction'] = pd.to_datetime(edges_df['first_interaction'], errors='coerce')
edges_df['last_interaction']  = pd.to_datetime(edges_df['last_interaction'],  errors='coerce')

# Remove redacted / unknown entities for a cleaner graph
NOISE = {'Redacted/Unknown', 'unknown', '', 'Unknown'}
edges_clean = edges_df[
    ~edges_df['source'].isin(NOISE) &
    ~edges_df['target'].isin(NOISE)
].copy()

print(f'Nodes:            {len(nodes_df):,}')
print(f'Raw edges:        {len(edges_df):,}')
print(f'Filtered edges:   {len(edges_clean):,}')
print(f'Date range:       {edges_df["first_interaction"].min().date()}  →  {edges_df["last_interaction"].max().date()}')

edges_clean.head()

In [ ]:
# ── 3. Build NetworkX Graph ───────────────────────────────────────────────────
G = nx.from_pandas_edgelist(
    edges_clean,
    source='source',
    target='target',
    edge_attr=['weight', 'first_interaction', 'last_interaction'],
    create_using=nx.Graph()
)

degree_dict   = dict(G.degree())
wdegree_dict  = dict(G.degree(weight='weight'))

# Giant connected component
comps     = sorted(nx.connected_components(G), key=len, reverse=True)
G_giant   = G.subgraph(comps[0]).copy()

degree_df = pd.DataFrame({
    'name':           list(degree_dict.keys()),
    'degree':         list(degree_dict.values()),
    'weighted_degree': [wdegree_dict.get(n, 0) for n in degree_dict]
}).sort_values('degree', ascending=False).reset_index(drop=True)

print(f'Nodes:                    {G.number_of_nodes():,}')
print(f'Edges:                    {G.number_of_edges():,}')
print(f'Connected components:     {nx.number_connected_components(G)}')
print(f'Giant component:          {G_giant.number_of_nodes():,} nodes / {G_giant.number_of_edges():,} edges')
print(f'Avg degree:               {np.mean(list(degree_dict.values())):.2f}')
print(f'Network density:          {nx.density(G):.5f}')
print()
print('Top 20 most connected nodes:')
display(degree_df.head(20))

## Visualization 1 — Interactive Full Network

- **Scroll** to zoom, **drag** to pan, **drag nodes** to reposition  
- **Hover** over any node or edge for details  
- Node **size** ∝ number of connections; **colour** encodes degree tier  
- 🔴 Jeffrey Epstein &nbsp;|&nbsp; 🟠 High degree (≥30) &nbsp;|&nbsp; 🟡 Medium (≥15) &nbsp;|&nbsp; 🟢 Low (≥3) &nbsp;|&nbsp; 🔵 Peripheral

In [ ]:
# ── 4. Interactive Network (pyvis) ────────────────────────────────────────────
net = Network(
    height='780px', width='100%',
    bgcolor='#0d1117', font_color='#e6edf3',
    notebook=True, cdn_resources='in_line'
)

net.set_options("""
{
  "nodes": { "borderWidth": 1, "shadow": {"enabled": true} },
  "edges": {
    "color": {"opacity": 0.35},
    "smooth": {"type": "dynamic"}
  },
  "physics": {
    "forceAtlas2Based": {
      "gravitationalConstant": -80,
      "centralGravity": 0.005,
      "springLength": 120,
      "springConstant": 0.08,
      "damping": 0.4,
      "avoidOverlap": 0.5
    },
    "maxVelocity": 50,
    "solver": "forceAtlas2Based",
    "timestep": 0.35,
    "stabilization": {"enabled": true, "iterations": 250, "updateInterval": 25}
  },
  "interaction": {
    "hover": true,
    "tooltipDelay": 80,
    "hideEdgesOnDrag": true,
    "navigationButtons": true,
    "keyboard": true
  }
}
""")

max_deg  = max(degree_dict.values()) or 1
max_wdeg = max(wdegree_dict.values()) or 1

def node_color(name, deg):
    if 'jeffrey epstein' in name.lower():
        return '#ff4444'
    if deg >= 30: return '#ff6b35'
    if deg >= 15: return '#f7931e'
    if deg >= 8:  return '#ffd700'
    if deg >= 3:  return '#69db7c'
    return '#74c0fc'

for node in G.nodes():
    deg  = degree_dict.get(node, 0)
    wdeg = wdegree_dict.get(node, 0)
    is_epstein = 'jeffrey epstein' in node.lower()
    size  = 55 if is_epstein else (8 + (deg / max_deg) * 45)
    color = node_color(node, deg)
    net.add_node(
        node, label=node, size=size, color=color,
        title=(
            "<div style='font-family:Arial;background:#1c1c2e;color:#e6edf3;"
            "padding:10px 12px;border-radius:8px;min-width:160px'>"
            f"<b style='font-size:14px'>{node}</b><br>"
            "<hr style='border:0;border-top:1px solid #444;margin:6px 0'>"
            f"Connections: <b>{deg}</b><br>"
            f"Total emails: <b>{wdeg}</b>"
            "</div>"
        )
    )

for u, v, data in G.edges(data=True):
    w     = data.get('weight', 1)
    first = data.get('first_interaction', '')
    last  = data.get('last_interaction', '')
    if isinstance(first, pd.Timestamp): first = first.strftime('%Y-%m-%d')
    if isinstance(last,  pd.Timestamp): last  = last.strftime('%Y-%m-%d')
    net.add_edge(
        u, v, value=min(w, 30),
        title=(
            "<div style='font-family:Arial;background:#1c1c2e;color:#e6edf3;"
            "padding:8px 10px;border-radius:6px'>"
            f"<b>{u}</b> ↔ <b>{v}</b><br>"
            f"Emails: <b>{w}</b><br>"
            f"First: {first}<br>Last: {last}"
            "</div>"
        )
    )

html = net.generate_html()
with open('jmail_network_graph.html', 'w', encoding='utf-8') as f:
    f.write(html)

display(HTML(html))

## Visualization 2 — Top Contacts of Jeffrey Epstein

In [ ]:
# ── 5. Top Contacts Bar Chart ─────────────────────────────────────────────────
epstein_edges = edges_df[
    (edges_df['source'] == 'Jeffrey Epstein') |
    (edges_df['target'] == 'Jeffrey Epstein')
].copy()

epstein_edges['contact'] = epstein_edges.apply(
    lambda r: r['target'] if r['source'] == 'Jeffrey Epstein' else r['source'], axis=1
)

top_contacts = (
    epstein_edges[~epstein_edges['contact'].isin(NOISE | {'Jeffrey Epstein'})]
    .groupby('contact')['weight'].sum()
    .sort_values(ascending=False)
    .head(35)
)

colors = ['#ff4444' if i == 0 else f'rgba(255,{80 + i*4},{30 + i*2},0.85)' for i in range(len(top_contacts))]

fig = go.Figure(go.Bar(
    x=top_contacts.values,
    y=top_contacts.index,
    orientation='h',
    marker=dict(color=list(reversed(colors))),
    text=top_contacts.values,
    textposition='outside',
    hovertemplate='<b>%{y}</b><br>Total emails: %{x}<extra></extra>'
))

fig.update_layout(
    title=dict(text='Top 35 Contacts of Jeffrey Epstein — Total Emails Exchanged', font_size=17),
    xaxis_title='Total Emails',
    yaxis=dict(autorange='reversed', tickfont_size=11),
    height=900,
    plot_bgcolor='#0d1117', paper_bgcolor='#0d1117',
    font_color='#e6edf3',
    margin=dict(l=200, r=80, t=70, b=50)
)
fig.update_xaxes(gridcolor='#2a2a2a')
fig.show()

## Visualization 3 — Activity Timeline

In [ ]:
# ── 6. Email Activity Timeline ────────────────────────────────────────────────
tl = edges_df.copy()
tl['year_first'] = tl['first_interaction'].dt.year
tl['year_last']  = tl['last_interaction'].dt.year

# New unique contacts per year (by first interaction)
new_per_year = (
    tl[~tl['source'].isin(NOISE) & ~tl['target'].isin(NOISE)]
    .groupby('year_first').size()
    .reset_index(columns=['year', 'new_connections'])
)
# Fix rename
new_per_year = (
    tl[~tl['source'].isin(NOISE) & ~tl['target'].isin(NOISE)]
    .groupby('year_first').size()
    .rename_axis('year').reset_index(name='new_connections')
)

# Total email volume per year
vol_per_year = (
    tl.groupby('year_first')['weight'].sum()
    .rename_axis('year').reset_index(name='total_emails')
)

# Cumulative unique contacts over time
new_per_year['cumulative'] = new_per_year['new_connections'].cumsum()

fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=[
        'New Connections Added Per Year',
        'Total Email Volume Per Year',
        'Cumulative Network Growth'
    ],
    vertical_spacing=0.1
)

fig.add_trace(go.Bar(
    x=new_per_year['year'], y=new_per_year['new_connections'],
    marker_color='#ff6b35', name='New Connections',
    hovertemplate='<b>%{x}</b><br>New connections: %{y}<extra></extra>'
), row=1, col=1)

fig.add_trace(go.Bar(
    x=vol_per_year['year'], y=vol_per_year['total_emails'],
    marker_color='#74c0fc', name='Email Volume',
    hovertemplate='<b>%{x}</b><br>Total emails: %{y}<extra></extra>'
), row=2, col=1)

fig.add_trace(go.Scatter(
    x=new_per_year['year'], y=new_per_year['cumulative'],
    mode='lines+markers',
    line=dict(color='#69db7c', width=2.5),
    marker=dict(size=6),
    name='Cumulative',
    hovertemplate='<b>%{x}</b><br>Total connections: %{y}<extra></extra>'
), row=3, col=1)

fig.update_layout(
    height=750, showlegend=False,
    plot_bgcolor='#0d1117', paper_bgcolor='#0d1117',
    font_color='#e6edf3',
    title=dict(text='Jmail Network — Activity Timeline', font_size=17)
)
for r in [1, 2, 3]:
    fig.update_xaxes(gridcolor='#2a2a2a', row=r, col=1)
    fig.update_yaxes(gridcolor='#2a2a2a', row=r, col=1)

fig.show()

## Visualization 4 — Degree Distribution

In [ ]:
# ── 7. Degree Distribution ────────────────────────────────────────────────────
deg_vals = sorted(degree_dict.values(), reverse=True)
deg_cnt  = Counter(deg_vals)
x_hist   = sorted(deg_cnt.keys())
y_hist   = [deg_cnt[x] for x in x_hist]

sorted_degs = sorted(deg_vals)
cdf_y = np.arange(1, len(sorted_degs) + 1) / len(sorted_degs)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Degree Distribution (log-log)', 'Cumulative Degree Distribution']
)

fig.add_trace(go.Bar(
    x=x_hist, y=y_hist,
    marker_color='#69db7c',
    hovertemplate='Degree: %{x}<br>Nodes: %{y}<extra></extra>'
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=sorted_degs, y=cdf_y,
    mode='lines', line=dict(color='#f7931e', width=2.5),
    hovertemplate='Degree: %{x}<br>Percentile: %{y:.1%}<extra></extra>'
), row=1, col=2)

fig.update_layout(
    height=420, showlegend=False,
    plot_bgcolor='#0d1117', paper_bgcolor='#0d1117',
    font_color='#e6edf3',
    title=dict(text='Network Degree Distribution', font_size=17)
)
fig.update_xaxes(type='log', gridcolor='#2a2a2a', row=1, col=1)
fig.update_yaxes(type='log', gridcolor='#2a2a2a', row=1, col=1)
fig.update_xaxes(gridcolor='#2a2a2a', row=1, col=2)
fig.update_yaxes(gridcolor='#2a2a2a', row=1, col=2)

fig.show()

leaf_pct = 100 * sum(1 for d in deg_vals if d == 1) / len(deg_vals)
print(f'Max degree:       {max(deg_vals)}')
print(f'Median degree:    {np.median(deg_vals):.1f}')
print(f'Mean degree:      {np.mean(deg_vals):.2f}')
print(f'Leaf nodes (d=1): {sum(1 for d in deg_vals if d == 1):,}  ({leaf_pct:.1f}%)')

## Visualization 5 — Community Detection

In [ ]:
# ── 8. Community Detection ────────────────────────────────────────────────────
from networkx.algorithms import community as nx_comm

try:
    communities = nx_comm.louvain_communities(G, weight='weight', seed=42)
    method = 'Louvain'
except AttributeError:
    communities = list(nx_comm.greedy_modularity_communities(G, weight='weight'))
    method = 'Greedy Modularity'

communities = sorted(communities, key=len, reverse=True)
node_to_comm = {node: i for i, comm in enumerate(communities) for node in comm}

print(f'Method:                {method}')
print(f'Total communities:     {len(communities)}')
print()

rows = []
for i, comm in enumerate(communities[:15]):
    top_members = sorted(comm, key=lambda n: degree_dict.get(n, 0), reverse=True)[:6]
    rows.append({'Community': f'C{i+1}', 'Size': len(comm), 'Top Members': ', '.join(top_members)})

comm_table = pd.DataFrame(rows)
display(comm_table)

# Bar chart of community sizes
top_sizes  = [len(c) for c in communities[:20]]
top_labels = [f'C{i+1}' for i in range(len(top_sizes))]

fig = go.Figure(go.Bar(
    x=top_labels, y=top_sizes,
    marker=dict(color=top_sizes, colorscale='Viridis', showscale=True,
                colorbar=dict(title='Size')),
    text=top_sizes, textposition='outside',
    hovertemplate='<b>%{x}</b><br>Members: %{y}<extra></extra>'
))

fig.update_layout(
    title=dict(text=f'Community Sizes — {method} (Top 20)', font_size=17),
    xaxis_title='Community', yaxis_title='Members',
    height=430,
    plot_bgcolor='#0d1117', paper_bgcolor='#0d1117',
    font_color='#e6edf3'
)
fig.update_xaxes(gridcolor='#2a2a2a')
fig.update_yaxes(gridcolor='#2a2a2a')
fig.show()

## Visualization 6 — Epstein Ego Network (Direct Circle)

Only nodes **directly connected** to Jeffrey Epstein. Nodes are coloured by community cluster.

In [ ]:
# ── 9. Ego Network ────────────────────────────────────────────────────────────
EPSTEIN = 'Jeffrey Epstein'

if EPSTEIN in G.nodes():
    ego_G = nx.ego_graph(G, EPSTEIN, radius=1)
else:
    candidates = [n for n in G.nodes() if 'epstein' in n.lower()]
    EPSTEIN = candidates[0] if candidates else list(G.nodes())[0]
    ego_G = nx.ego_graph(G, EPSTEIN, radius=1)

print(f'Ego network ({EPSTEIN}): {ego_G.number_of_nodes()} nodes, {ego_G.number_of_edges()} edges')

ego_deg  = dict(ego_G.degree())
ego_wdeg = dict(ego_G.degree(weight='weight'))
max_edeg = max(ego_deg.values()) or 1

PALETTE = [
    '#ff6b35','#f7931e','#ffd700','#69db7c','#74c0fc',
    '#da77f2','#ff8787','#a9e34b','#63e6be','#e599f7',
    '#ffa8a8','#66d9e8','#b2f2bb','#ffec99','#d0bfff'
]

ego_net = Network(
    height='700px', width='100%',
    bgcolor='#0d1117', font_color='#e6edf3',
    notebook=True, cdn_resources='in_line'
)

ego_net.set_options("""
{
  "physics": {
    "barnesHut": {
      "gravitationalConstant": -8000,
      "centralGravity": 0.3,
      "springLength": 130,
      "springConstant": 0.04,
      "damping": 0.09
    }
  },
  "interaction": {
    "hover": true,
    "tooltipDelay": 80,
    "navigationButtons": true
  }
}
""")

for node in ego_G.nodes():
    deg  = ego_deg.get(node, 0)
    wdeg = ego_wdeg.get(node, 0)
    is_ep = 'jeffrey epstein' in node.lower()
    comm  = node_to_comm.get(node, 0)
    color = '#ff4444' if is_ep else PALETTE[comm % len(PALETTE)]
    size  = 55 if is_ep else (10 + (deg / max_edeg) * 35)
    ego_net.add_node(
        node, label=node, size=size, color=color,
        title=(
            "<div style='font-family:Arial;background:#1c1c2e;color:#e6edf3;"
            "padding:10px 12px;border-radius:8px'>"
            f"<b style='font-size:13px'>{node}</b><br>"
            "<hr style='border:0;border-top:1px solid #444;margin:5px 0'>"
            f"Connections (ego): <b>{deg}</b><br>"
            f"Total emails: <b>{wdeg}</b><br>"
            f"Community: <b>C{comm + 1}</b>"
            "</div>"
        )
    )

for u, v, data in ego_G.edges(data=True):
    w = data.get('weight', 1)
    ego_net.add_edge(u, v, value=min(w, 20))

ego_html = ego_net.generate_html()
with open('epstein_ego_network.html', 'w', encoding='utf-8') as f:
    f.write(ego_html)

display(HTML(ego_html))

## Visualization 7 — Connection Strength Heatmap (Top Contacts)

In [ ]:
# ── 10. Heatmap of Top Contact Interactions ───────────────────────────────────
N = 30
top_n = top_contacts.head(N).index.tolist()
nodes_in_heatmap = ['Jeffrey Epstein'] + top_n

sub = edges_df[
    edges_df['source'].isin(nodes_in_heatmap) &
    edges_df['target'].isin(nodes_in_heatmap)
]

matrix = pd.DataFrame(0, index=nodes_in_heatmap, columns=nodes_in_heatmap)
for _, row in sub.iterrows():
    if row['source'] in matrix.index and row['target'] in matrix.columns:
        matrix.loc[row['source'], row['target']] += row['weight']
        matrix.loc[row['target'], row['source']] += row['weight']

fig = go.Figure(go.Heatmap(
    z=matrix.values,
    x=matrix.columns.tolist(),
    y=matrix.index.tolist(),
    colorscale='Reds',
    hovertemplate='%{y} ↔ %{x}<br>Emails: %{z}<extra></extra>',
    colorbar=dict(title='Emails')
))

fig.update_layout(
    title=dict(text='Email Volume Heatmap — Jeffrey Epstein & Top 30 Contacts', font_size=17),
    height=780,
    plot_bgcolor='#0d1117', paper_bgcolor='#0d1117',
    font_color='#e6edf3',
    xaxis=dict(tickangle=50, tickfont_size=10),
    yaxis=dict(tickfont_size=10),
    margin=dict(l=180, r=60, b=160, t=70)
)
fig.show()

## Visualization 8 — Betweenness Centrality (Bridge Nodes)

In [ ]:
# ── 11. Betweenness Centrality ────────────────────────────────────────────────
# Nodes that act as bridges / information brokers in the network
# Computed on the giant component only (faster)
print('Computing betweenness centrality (may take a moment)...')
btw = nx.betweenness_centrality(G_giant, weight='weight', normalized=True)
btw_df = pd.DataFrame(list(btw.items()), columns=['node', 'betweenness'])
btw_df = btw_df.sort_values('betweenness', ascending=False).reset_index(drop=True)

print('Top 20 bridge nodes (highest betweenness centrality):')
display(btw_df.head(20))

# Plot top 25
top_btw = btw_df.head(25)

fig = go.Figure(go.Bar(
    x=top_btw['betweenness'],
    y=top_btw['node'],
    orientation='h',
    marker=dict(color=top_btw['betweenness'], colorscale='Plasma', reversescale=True),
    hovertemplate='<b>%{y}</b><br>Betweenness: %{x:.5f}<extra></extra>'
))

fig.update_layout(
    title=dict(text='Top 25 Bridge Nodes — Betweenness Centrality', font_size=17),
    xaxis_title='Betweenness Centrality (normalised)',
    yaxis=dict(autorange='reversed', tickfont_size=11),
    height=700,
    plot_bgcolor='#0d1117', paper_bgcolor='#0d1117',
    font_color='#e6edf3',
    margin=dict(l=200, r=60, t=70, b=50)
)
fig.update_xaxes(gridcolor='#2a2a2a')
fig.show()

## Visualization 9 — Filterable Ego Network (Depth-2)

Explore any node's 2-hop neighbourhood — change `FOCUS_NODE` to any name in the network.

In [ ]:
# ── 12. Custom Ego Network Explorer ──────────────────────────────────────────
# ★ Change this to any node in the network ★
FOCUS_NODE = 'Jeffrey Epstein'
RADIUS     = 2   # 1 = direct contacts only, 2 = friends-of-friends

if FOCUS_NODE not in G.nodes():
    matches = [n for n in G.nodes() if FOCUS_NODE.lower() in n.lower()]
    print(f'Exact match not found. Closest: {matches[:5]}')
    FOCUS_NODE = matches[0] if matches else list(G.nodes())[0]

sub_G = nx.ego_graph(G, FOCUS_NODE, radius=RADIUS)
print(f'Ego network of "{FOCUS_NODE}" (radius={RADIUS}): {sub_G.number_of_nodes()} nodes, {sub_G.number_of_edges()} edges')

sub_deg  = dict(sub_G.degree())
sub_wdeg = dict(sub_G.degree(weight='weight'))
max_sdeg = max(sub_deg.values()) or 1

sub_net = Network(
    height='720px', width='100%',
    bgcolor='#0d1117', font_color='#e6edf3',
    notebook=True, cdn_resources='in_line'
)
sub_net.set_options("""
{
  "physics": {
    "forceAtlas2Based": {
      "gravitationalConstant": -60,
      "centralGravity": 0.01,
      "springLength": 110,
      "springConstant": 0.08
    },
    "solver": "forceAtlas2Based",
    "stabilization": {"iterations": 200}
  },
  "interaction": {"hover": true, "navigationButtons": true}
}
""")

# Colour by hop distance from focus node
hop_colors = {0: '#ff4444', 1: '#ff6b35', 2: '#74c0fc'}

for node in sub_G.nodes():
    hop  = nx.shortest_path_length(sub_G, FOCUS_NODE, node) if node != FOCUS_NODE else 0
    deg  = sub_deg.get(node, 0)
    wdeg = sub_wdeg.get(node, 0)
    size = 50 if node == FOCUS_NODE else (8 + (deg / max_sdeg) * 35)
    color = hop_colors.get(hop, '#aaa')
    sub_net.add_node(
        node, label=node, size=size, color=color,
        title=(
            "<div style='font-family:Arial;background:#1c1c2e;color:#e6edf3;"
            "padding:10px;border-radius:8px'>"
            f"<b>{node}</b><br>"
            f"Hop: {hop} &nbsp;|&nbsp; Degree: {deg}<br>"
            f"Total emails: {wdeg}"
            "</div>"
        )
    )

for u, v, data in sub_G.edges(data=True):
    sub_net.add_edge(u, v, value=min(data.get('weight', 1), 20))

sub_html = sub_net.generate_html()
with open('ego_explorer.html', 'w', encoding='utf-8') as f:
    f.write(sub_html)

display(HTML(f'<h3 style="color:#e6edf3;background:#0d1117;padding:10px">Ego Network: {FOCUS_NODE} (radius={RADIUS})</h3>'))
display(HTML('<p style="color:#888">🔴 Focus node &nbsp;|&nbsp; 🟠 1-hop &nbsp;|&nbsp; 🔵 2-hop</p>'))
display(HTML(sub_html))

---
## Summary Statistics

In [ ]:
# ── 13. Summary ───────────────────────────────────────────────────────────────
epstein_deg  = degree_dict.get('Jeffrey Epstein', 0)
epstein_wdeg = wdegree_dict.get('Jeffrey Epstein', 0)

print('=' * 55)
print('  JMAIL NETWORK — SUMMARY')
print('=' * 55)
print(f'  Total unique entities:      {G.number_of_nodes():,}')
print(f'  Total unique connections:   {G.number_of_edges():,}')
print(f'  Date range:                 {edges_df["first_interaction"].min().strftime("%Y-%m-%d")} → {edges_df["last_interaction"].max().strftime("%Y-%m-%d")}')
print()
print(f'  Jeffrey Epstein')
print(f'    Direct contacts:          {epstein_deg:,}')
print(f'    Total emails:             {epstein_wdeg:,}')
print(f'    Betweenness rank:         #{(btw_df["node"] == "Jeffrey Epstein").idxmax() + 1 if "Jeffrey Epstein" in btw_df["node"].values else "N/A"}')
print()
print(f'  Communities detected:       {len(communities)}')
print(f'  Largest community:          {len(communities[0]):,} members')
print(f'  Network density:            {nx.density(G):.5f}')
print(f'  Avg clustering coefficient: {nx.average_clustering(G):.4f}')
print('=' * 55)